# Full read download on Colab, one accession at a time
Total for all 8 accessions: **25.7 GB**, which is more than free Google Drive (15 GB) can hold at once.
So each accession has **its own cell**. The workflow for each one:

1. Run the accession's cell. It downloads R1 + R2 to Colab's disk, checks the **md5** against ENA,
   then copies the verified files to `My Drive/genom_project/raw_full/`.
2. Download those two files from Google Drive to your laptop, into `d:\genom_project\data\raw_full\`.
3. Run the **Free Drive space** cell at the bottom, then **empty the Drive trash**
   (drive.google.com > Trash > Empty trash). Deleted files still count against your quota until the trash is emptied.
4. Go to the next accession.

Run the two **Setup** cells first, every time the Colab session restarts.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Setup: download helper (run once per session)
import os, subprocess, time, shutil
!apt-get -qq install -y aria2 > /dev/null && echo aria2 ready

LOCAL = '/content/raw_full'
DRIVE = '/content/drive/MyDrive/genom_project/raw_full'
os.makedirs(LOCAL, exist_ok=True); os.makedirs(DRIVE, exist_ok=True)

# sample: (run, md5 R1, md5 R2, bytes R1, bytes R2)
RUNS = {
    'Col-0': ('SRR1945757', '2ee4ac72645d55819431900d74fa7386', 'cb1e27d31c144ea03ab3a62888a8ec41', 1716633524, 1732237033),
    'An-1': ('SRR1945750', '137e0905348fe5d135301240c2e5ebdb', '4017e24879047b390a8a8a81b63a1575', 777193465, 788816817),
    'Cvi-0': ('SRR1945758', '5a160c6fe5f08586c678dc4c9644e8ba', '3ed2692a703a4764af6b575ce1c35192', 3462708649, 3258948146),
    'Ler-1': ('SRR1945772', 'c41060d8bb7cf2c75d76f16e571555eb', '979d626d86b499c35c917c8ee792d51f', 2194455309, 2390999044),
    'Kyoto': ('SRR1945866', 'ec2a413fc8fd981662101cc4ad505eb5', '65e8d4ffa75bccf6c971048ddc1a04c4', 894350704, 877964798),
    'Ale-Stenar-44-4': ('SRR1945477', 'e3bbf1f76b865dbc554641fbf00e68b5', 'ea9a62572028d1488dccf07ea08fb276', 1552344058, 1614458369),
    'Se-0': ('SRR1945785', '07962f1be69dd9fd4ac8da0e6c1d89ea', '39b5644cbae7480f17060c208d4af5bc', 1173323224, 1164047295),
    'Kondara': ('SRR1945770', '8d842186fc9575bc124fb7065ab9bec7', 'eba5af85fcfffd60f737460a11b5b662', 1053657328, 1084289228),
}

def url(run, r):
    return f'https://ftp.sra.ebi.ac.uk/vol1/fastq/{run[:6]}/00{run[-1]}/{run}/{run}_{r}.fastq.gz'

def md5(path):
    return subprocess.run(['md5sum', path], capture_output=True, text=True).stdout.split()[0]

def get(sample):
    run, m1, m2, b1, b2 = RUNS[sample]
    need = (b1 + b2) / 1e9
    free = shutil.disk_usage('/content').free / 1e9
    print(f'{sample} ({run}): {need:.2f} GB to download, Colab disk free {free:.0f} GB')
    for r, want in ((1, m1), (2, m2)):
        name = f'{sample}_R{r}.fastq.gz'
        local, final = f'{LOCAL}/{name}', f'{DRIVE}/{name}'
        if os.path.exists(final) and os.path.exists(final + '.md5ok'):
            print(f'  skip {name}: already verified on Drive'); continue
        for attempt in range(1, 6):
            t0 = time.time()
            # 8 connections per file, resumes automatically if interrupted
            subprocess.run(['aria2c', '-q', '-x8', '-s8', '-c', '--max-tries=10', '--retry-wait=5',
                            '-d', LOCAL, '-o', name, url(run, r)])
            got = md5(local) if os.path.exists(local) else ''
            if got == want:
                mb = os.path.getsize(local) / 1e6
                print(f'  md5 OK  {name}  {mb:,.0f} MB in {time.time() - t0:.0f}s, copying to Drive...')
                shutil.copy(local, final)
                if md5(final) != want:
                    raise RuntimeError(f'{name}: copy on Drive is corrupt, run the cell again')
                open(final + '.md5ok', 'w').write(want + '\n')
                os.remove(local)
                break
            print(f'  attempt {attempt}: md5 mismatch for {name} (got {got or "no file"}), retrying')
            if os.path.exists(local): os.remove(local)
        else:
            raise RuntimeError(f'{name} failed after 5 attempts')
    print(f'DONE {sample}: download both files from My Drive/genom_project/raw_full/ to your laptop')

def free_drive(sample):
    for r in (1, 2):
        for ext in ('', '.md5ok'):
            p = f'{DRIVE}/{sample}_R{r}.fastq.gz{ext}'
            if os.path.exists(p): os.remove(p); print('deleted', p)
    print('Now empty the Drive trash (drive.google.com > Trash) to get the space back.')

print('ready:', ', '.join(RUNS))

## One cell per accession
Run one, move the files to your laptop, free Drive space, then run the next.

### Col-0 (SRR1945757), 3.45 GB

In [ ]:
get('Col-0')

### An-1 (SRR1945750), 1.57 GB

In [ ]:
get('An-1')

### Cvi-0 (SRR1945758), 6.72 GB

In [ ]:
get('Cvi-0')

### Ler-1 (SRR1945772), 4.59 GB

In [ ]:
get('Ler-1')

### Kyoto (SRR1945866), 1.77 GB

In [ ]:
get('Kyoto')

### Ale-Stenar-44-4 (SRR1945477), 3.17 GB

In [ ]:
get('Ale-Stenar-44-4')

### Se-0 (SRR1945785), 2.34 GB

In [ ]:
get('Se-0')

### Kondara (SRR1945770), 2.14 GB

In [ ]:
get('Kondara')

## Free Drive space
Run this **only after** both files of that accession are safely on your laptop. Change the name to the accession you just moved.

In [ ]:
free_drive('Col-0')   # <- change to the accession you have already copied to your laptop

## On your laptop afterwards
Put the files in `d:\genom_project\data\raw_full\`, then ask Claude to verify them. Each file is checked against the same md5 values.